This notebook will:
✅ create embeddings
✅ store vectors in ChromaDB
✅ save persistent vector DB

Then Notebook 07 will work.

In [1]:
# Imports
import os
import re

import chromadb
import pandas as pd

from sentence_transformers import SentenceTransformer

In [2]:
#Load Dataset
airline_df = pd.read_csv(
    "../data/raw/Twitter US Airline Sentiment/Tweets.csv"
)

print("Dataset Loaded!")

Dataset Loaded!


In [3]:
# Cleaning Function
def clean_tweet(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
# Clean Tweets
airline_df["clean_text"] = airline_df[
    "text"
].apply(clean_tweet)

print("Cleaning Complete!")

Cleaning Complete!


In [5]:
# Create RAG Documents
airline_df["rag_document"] = (

    "Airline: " +

    airline_df["airline"].astype(str)

    +

    "\nSentiment: " +

    airline_df["airline_sentiment"].astype(str)

    +

    "\nIssue: " +

    airline_df["negativereason"].fillna(
        "No Issue"
    ).astype(str)

    +

    "\nCustomer Message: " +

    airline_df["clean_text"]
)

In [6]:
# Load Embedding Model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [7]:
# Create ChromaDB Client
chroma_client = chromadb.PersistentClient(
    path="../vector_db"
)

collection = chroma_client.get_or_create_collection(
    name="airline_support_knowledge"
)

print("ChromaDB ready!")

ChromaDB ready!


In [8]:
# Generate Embeddings
documents = airline_df[
    "rag_document"
].tolist()

embeddings = embedding_model.encode(
    documents
).tolist()

print("Embeddings generated!")

Embeddings generated!


In [10]:
# Store In ChromaDB
# Store In ChromaDB Using Batches

batch_size = 5000

for i in range(0, len(documents), batch_size):

    batch_documents = documents[i:i+batch_size]

    batch_embeddings = embeddings[i:i+batch_size]

    batch_ids = [
        str(x)
        for x in range(i, i + len(batch_documents))
    ]

    collection.add(
        documents=batch_documents,
        embeddings=batch_embeddings,
        ids=batch_ids
    )

    print(f"Batch {i} stored successfully!")

print("Vector database created successfully!")

Batch 0 stored successfully!
Batch 5000 stored successfully!
Batch 10000 stored successfully!
Vector database created successfully!


In [11]:
# Test Vector DB
results = collection.query(
    query_embeddings=[
        embeddings[0]
    ],
    n_results=2
)

results["documents"][0]

['Airline: Virgin America\nSentiment: neutral\nIssue: No Issue\nCustomer Message: what said',
 'Airline: Virgin America\nSentiment: neutral\nIssue: No Issue\nCustomer Message: good point']